# NYC 311 Service Requests (2024) — Cleaning & EDA

**Business question:** *Which complaint types and boroughs had the slowest resolution
times in 2024, and how should the city reallocate response resources?*

**Core metric — Resolution Time:** `closed_date − created_date`, in **hours**.

This notebook takes the raw ~2 GB / 3.46M-row export and produces a clean, analysis-ready
`data/311_cleaned_2024.csv`. Every cleaning step below is paired with a short note on the
*judgment* behind it — not just the code — because that reasoning is the deliverable.

It runs top-to-bottom: raw CSV → cleaned CSV.

## 0. Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__)

# Repo-root-relative paths so the notebook runs from anywhere.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_CSV = ROOT / "data" / "raw" / "nyc_311_2024.csv"
OUT_CSV = ROOT / "data" / "311_cleaned_2024.csv"
assert RAW_CSV.exists(), f"Raw file not found: {RAW_CSV}"
print("raw :", RAW_CSV)
print("out :", OUT_CSV)

pandas 1.5.3 | numpy 1.26.4
raw : C:\01_Projects\analysis\data\raw\nyc_311_2024.csv
out : C:\01_Projects\analysis\data\311_cleaned_2024.csv


## 1. Load the raw data (only the columns we need)

The raw file is ~2 GB with 44 columns and 3.46M rows. Reading all of it is slow and
wasteful, so we load **only the 10 columns** the resolution-time analysis actually uses
(see `data/data_dictionary.md` for the full keep/drop rationale) and push as much work as
possible into the read itself:

- **`usecols`** — read 10 of 44 columns.
- **`parse_dates`** — parse the two timestamps at read time (they drive the core metric).
- **`dtype='category'`** for low-cardinality strings (`agency`, `complaint_type`, `borough`,
  `status`) to cut memory.
- **`incident_zip` as string** — the raw file stores ZIPs as floats (e.g. `11226.0`); reading
  as string avoids re-introducing that float artifact and a fake numeric meaning.

In [2]:
USECOLS = [
    "unique_key", "created_date", "closed_date", "agency", "complaint_type",
    "descriptor", "borough", "incident_zip", "status", "resolution_description",
]
DTYPES = {
    "agency": "category", "complaint_type": "category",
    "borough": "category", "status": "category", "incident_zip": "string",
}

df = pd.read_csv(
    RAW_CSV,
    usecols=USECOLS,
    dtype=DTYPES,
    parse_dates=["created_date", "closed_date"],
    infer_datetime_format=True,
)

print(f"loaded {len(df):,} rows x {df.shape[1]} cols")
print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
df.dtypes

loaded 3,456,770 rows x 10 cols


memory: 1,335 MB


unique_key                         int64
created_date              datetime64[ns]
closed_date               datetime64[ns]
agency                          category
complaint_type                  category
descriptor                        object
incident_zip                      string
status                          category
resolution_description            object
borough                         category
dtype: object

## 2. Sanity-check the dates

The two timestamps were parsed to `datetime64` at load. We confirm the dtype and that
`created_date` stays within calendar-year 2024 — the project's scope. A stray 2023/2025
value would signal an extraction bug upstream.

In [3]:
print("created_date dtype:", df["created_date"].dtype)
print("closed_date  dtype:", df["closed_date"].dtype)
print("created_date range:", df["created_date"].min(), "->", df["created_date"].max())
print("closed_date  range:", df["closed_date"].min(), "->", df["closed_date"].max())
assert df["created_date"].dt.year.between(2024, 2024).all(), "created_date outside 2024"

created_date dtype: datetime64[ns]
closed_date  dtype: datetime64[ns]
created_date range: 2024-01-01 00:00:00 -> 2024-12-31 23:59:38
closed_date  range: 2023-02-22 10:15:00 -> 2026-06-19 08:04:55


## 3. Derive the core metric — `resolution_time_hours`

`resolution_time_hours = (closed_date − created_date)` in hours. It is **undefined (NaN)**
for any ticket without a `closed_date` (still open / unresolved) — we deal with those next.

In [4]:
df["resolution_time_hours"] = (
    (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
)
df["resolution_time_hours"].describe()

count    3.392221e+06
mean     3.222189e+02
std      1.202557e+03
min     -8.784017e+03
25%      9.866667e-01
50%      8.103889e+00
75%      8.353972e+01
max      2.126847e+04
Name: resolution_time_hours, dtype: float64

## 4. Open / unresolved tickets (null `closed_date`)

A ticket with no `closed_date` has no resolution time. The PRD anticipated these as
`status = 'Open'`, but the data is messier: **most** statuses can lack a close date — and
~27k rows are even marked `Closed` with no timestamp. So we key the rule on the **null
date itself, not the status string**.

**Decision:** keep these rows for *volume* counts (they're real complaints) but they
naturally carry `NaN` resolution time and are excluded from any resolution-time stat. We do
**not** drop them here — dropping would understate complaint volume.

In [5]:
n_null = df["closed_date"].isna().sum()
print(f"null closed_date: {n_null:,} ({n_null / len(df) * 100:.2f}% of all tickets)\n")
print("status breakdown of the null-closed rows:")
print(df.loc[df["closed_date"].isna(), "status"].value_counts(dropna=False))

null closed_date: 64,549 (1.87% of all tickets)

status breakdown of the null-closed rows:
Closed         27573
In Progress    26902
Open            5211
Pending         2758
Assigned        1377
Started          725
Unspecified        3
Name: status, dtype: int64


## 5. Drop negative resolution times (data-entry errors)

A negative resolution time means the ticket was closed *before* it was created — physically
impossible, so these are data-entry errors. They're a tiny fraction and there's no
defensible way to repair the true value, so we **drop the rows entirely** and flag the count.

In [6]:
neg_mask = df["resolution_time_hours"] < 0
n_neg = int(neg_mask.sum())
before = len(df)
df = df[~neg_mask].copy()
print(f"dropped {n_neg:,} rows with negative resolution time")
print(f"rows: {before:,} -> {len(df):,}")

dropped 988 rows with negative resolution time
rows: 3,456,770 -> 3,455,782


## 6. Cap extreme outliers at the 99th percentile

The resolution-time tail is extreme — the max is ~886 days. A handful of tickets sitting
open for years (or closed by a bulk administrative sweep) would otherwise dominate every
average. Following the PRD, we **winsorize**: values above the 99th percentile are clipped
*down to* that threshold rather than dropped, so the rows still count toward volume and
their boroughs/complaint types while their extreme magnitude stops skewing the means.

The threshold is computed from the data (the 99th percentile of valid, non-null resolution
times) and printed so the choice is transparent and reproducible.

In [7]:
cap = df["resolution_time_hours"].quantile(0.99)
n_above = int((df["resolution_time_hours"] > cap).sum())
print(f"99th-percentile cap = {cap:,.1f} hours ({cap / 24:,.1f} days)")
print(f"clipping {n_above:,} rows above the cap")
print(f"max before: {df['resolution_time_hours'].max():,.1f} h", end="  ->  ")
df["resolution_time_hours"] = df["resolution_time_hours"].clip(upper=cap)
print(f"max after: {df['resolution_time_hours'].max():,.1f} h")

99th-percentile cap = 6,960.9 hours (290.0 days)
clipping 33,913 rows above the cap
max before: 21,268.5 h  ->  max after: 6,960.9 h


## 7. Standardise `borough`

The raw values are ALL-CAPS (`BROOKLYN`, `STATEN ISLAND`, …) plus `Unspecified` for missing
geography. We normalise to Title Case for clean dashboard labels and fold any blank/unknown
value into an explicit `Unspecified` bucket rather than dropping it — geography-less tickets
are still valid complaints.

In [8]:
print("before:")
print(df["borough"].value_counts(dropna=False), "\n")

VALID_BOROUGHS = {"Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"}
borough = df["borough"].astype("string").str.strip().str.title()
borough = borough.where(borough.isin(VALID_BOROUGHS), other="Unspecified")
df["borough"] = borough.astype("category")

print("after:")
print(df["borough"].value_counts(dropna=False))

before:
BROOKLYN         1044999
QUEENS            826668
BRONX             737655
MANHATTAN         722653
STATEN ISLAND     121173
Unspecified         2634
Name: borough, dtype: int64 



after:
Brooklyn         1044999
Queens            826668
Bronx             737655
Manhattan         722653
Staten Island     121173
Unspecified         2634
Name: borough, dtype: int64


## 8. Standardise `complaint_type`

The main inconsistency here is **casing**, not meaning: the same dataset carries
`HEAT/HOT WATER`, `UNSANITARY CONDITION` (all-caps, mostly HPD) alongside `Illegal Parking`,
`Noise - Residential` (title-case). We normalise casing with Title Case, which collapses
pure casing-duplicates.

**Judgment call — we do *not* over-merge.** The `Noise - Residential` / `Noise -
Commercial` / `Noise - Street/Sidewalk` / `Noise - Vehicle` split is a *meaningful*
operational distinction (different agencies, different fixes), so we deliberately keep those
as separate categories instead of bucketing them all into "Noise".

In [9]:
print(f"distinct complaint_type before: {df['complaint_type'].nunique()}")
df["complaint_type"] = (
    df["complaint_type"].astype("string").str.strip().str.title().astype("category")
)
print(f"distinct complaint_type after : {df['complaint_type'].nunique()}")
print("\ntop 10 after standardising:")
print(df["complaint_type"].value_counts().head(10))

distinct complaint_type before: 197


distinct complaint_type after : 195

top 10 after standardising:
Illegal Parking            505727
Noise - Residential        379251
Heat/Hot Water             264750
Blocked Driveway           170190
Noise - Street/Sidewalk    162996
Unsanitary Condition       120854
Street Condition            71146
Abandoned Vehicle           70326
Plumbing                    69050
Noise - Commercial          68335
Name: complaint_type, dtype: int64


## 9. Clean `incident_zip`

The raw export stored ZIP codes as floats, so they arrive as `11226.0`. ZIP is an
identifier, not a number, so we strip the trailing `.0` and keep it as a string. (NYC ZIPs
have no leading zeros, so no information is lost.) Blank/missing ZIPs stay null.

In [10]:
df["incident_zip"] = (
    df["incident_zip"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
)
print(df["incident_zip"].head(5).tolist())
print("null ZIPs:", df["incident_zip"].isna().sum())

['11226', '11217', '11222', '11361', '11103']
null ZIPs: 34181


## 9b. Classify the resolution *outcome* — not just the speed

A quarter of closed tickets close in **under an hour**, but `resolution_description` shows many
of these (and many slower ones) **fixed nothing**: the agency found **no violation**, the party
was **gone on arrival**, the request was **referred** elsewhere, or it was a **duplicate**.
Counting those as "resolutions" would make enforcement-style complaints look hyper-efficient and
could drive a wrong *"reallocate away from NYPD"* conclusion.

So we **keep every row** (dropping ~10% of closed tickets would bias volume and is hard to
defend) and tag each with a derived **`resolution_category`**, parsed from the standardised
agency templates into five outcomes:

| Category | Meaning | Substantive? |
|---|---|:--:|
| `Action/Enforcement` | fixed / repaired / cleaned / collected / summons / violation issued | ✅ |
| `No issue found` | inspected; no violation/condition or no action necessary | — |
| `Gone/No access` | party gone on arrival, or couldn't gain entry to inspect | — |
| `Referred/Info` | sent elsewhere, duplicate, info-only, status redirect | — |
| `Open/Unresolved` | still open (null close date) | — |

**Rule order matters** — the "nothing found" patterns are matched *before* the action ones, so
HPD's *"No violations were issued"* isn't mistaken for *"Violations were issued"*. A ~17% long
tail of agency-specific wording is left as `Other` rather than force-fit into a class it doesn't
clearly belong to.

In [11]:
import numpy as np

# Classify each ticket from the standardised agency response templates.
# Order matters (np.select takes the FIRST match): the "nothing found / no action"
# patterns are tested BEFORE the action patterns, so HPD's "No violations were issued"
# is not mis-read as the substantive "Violations were issued".
desc = df["resolution_description"].fillna("").str.lower()
is_open = df["resolution_time_hours"].isna()

conds = [
    is_open,
    desc.str.contains(r"no violation|no evidence|did not violate|didn'?t observe|did not observe|"
                      r"found no (?:violation|condition)|no condition at|no further action|"
                      r"not necessary|no action was taken|did not require", regex=True, na=False),
    desc.str.contains(r"\bgone\b|unable to|could not gain entry|no access|attempted to conduct|"
                      r"not able to gain", regex=True, na=False),
    desc.str.contains(r"took action to fix|issued a summons|summons|violation|cleaned the location|"
                      r"collected the requested|repaired|corrected|completed the requested", regex=True, na=False),
    desc.str.contains(r"provided additional information|available on the department|please click|"
                      r"mailed you|duplicate|jurisdiction|submitted to|was referred|transferred to", regex=True, na=False),
]
choices = ["Open/Unresolved", "No issue found", "Gone/No access", "Action/Enforcement", "Referred/Info"]
df["resolution_category"] = pd.Categorical(np.select(conds, choices, default="Other"))

# "Substantive" = the agency actually fixed or enforced something.
SUBSTANTIVE = {"Action/Enforcement"}
closed_cat = df.loc[~is_open, "resolution_category"]
print(df["resolution_category"].value_counts(dropna=False))
print(f"\nsubstantive-resolution rate (of closed): {closed_cat.isin(SUBSTANTIVE).mean() * 100:.1f}%")
print(f"unclassified 'Other' (of closed):        {(closed_cat == 'Other').mean() * 100:.1f}%")

No issue found        1060626
Action/Enforcement     984341
Other                  591590
Gone/No access         432444
Referred/Info          322232
Open/Unresolved         64549
Name: resolution_category, dtype: int64

substantive-resolution rate (of closed): 29.0%
unclassified 'Other' (of closed):        17.4%


## 10. Select the final columns

We keep the 10 analysis columns plus the two derived fields (`resolution_time_hours`,
`resolution_category`) — **12 total** — in a sensible order. Everything else was dropped at
read time; see `data/data_dictionary.md`.

In [12]:
FINAL_COLS = [
    "unique_key", "created_date", "closed_date", "agency", "complaint_type",
    "descriptor", "borough", "incident_zip", "status", "resolution_description",
    "resolution_time_hours", "resolution_category",
]
df = df[FINAL_COLS]
df.dtypes

unique_key                         int64
created_date              datetime64[ns]
closed_date               datetime64[ns]
agency                          category
complaint_type                  category
descriptor                        object
borough                         category
incident_zip                      string
status                          category
resolution_description            object
resolution_time_hours            float64
resolution_category             category
dtype: object

## 11. Final validation

A few guard-rail checks before export: no negative resolution times survive, the cap held,
`borough` is fully standardised, and `unique_key` is unique. These will raise if a future
re-run regresses.

In [13]:
res = df["resolution_time_hours"]
assert res.min() >= 0, "negative resolution time leaked through"
assert res.max() <= cap + 1e-6, "value above the cap leaked through"
assert set(df["borough"].unique()) <= (VALID_BOROUGHS | {"Unspecified"}), "stray borough"

dupes = int(df["unique_key"].duplicated().sum())
print(f"final shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"duplicate unique_key: {dupes:,}")
print(f"tickets with a resolution time: {res.notna().sum():,}")
print(f"open/unresolved (NaN resolution): {res.isna().sum():,}")
print("\nnull counts per column:")
print(df.isna().sum())

final shape: 3,455,782 rows x 12 cols
duplicate unique_key: 0
tickets with a resolution time: 3,391,233
open/unresolved (NaN resolution): 64,549

null counts per column:


unique_key                     0
created_date                   0
closed_date                64549
agency                         0
complaint_type                 0
descriptor                112291
borough                        0
incident_zip               34181
status                         0
resolution_description     54115
resolution_time_hours      64549
resolution_category            0
dtype: int64


## 13. Analytical caveats (read before building visuals)

Three things the SQL / dashboard layer must respect, recorded here so the reasoning is on file:

1. **Lead with the median, not the mean.** Resolution time is extremely right-skewed
   (mean ≈ 298 h vs median ≈ 8 h). The mean is dominated by the long tail and the 99th-
   percentile cap; the median reflects the typical citizen's experience. Use median as the
   headline, mean as secondary with this caveat.
2. **Cap artifact.** Chronically slow categories (e.g. *New Tree Request* — seasonal Parks
   planting) have medians sitting **on** the 290-day cap, i.e. >50% of their tickets were
   clipped. Report those as ">cap (long-cycle)", not a precise number. They are listed below.
3. **Speed ≠ effectiveness.** Pair every resolution-time visual with the substantive-
   resolution rate (`resolution_category`) so fast-but-empty closes stay visible.

In [14]:
res = df["resolution_time_hours"]
closed = df[res.notna()]
print(f"citywide mean = {res.mean():,.1f} h    median = {res.median():,.1f} h    (skew -> lead with median)\n")

# Complaint types whose MEDIAN is pinned on the cap => >50% of their tickets were clipped.
ct = closed.groupby("complaint_type", observed=True)["resolution_time_hours"].agg(["count", "median"])
pinned = ct[(ct["count"] >= 500) & (ct["median"] >= cap * 0.999)].sort_values("median", ascending=False)
print(f"complaint types with median pinned at the {cap:,.0f} h cap (report as '>cap, long-cycle'):")
print(pinned.round(1).to_string() if len(pinned) else "  (none)", "\n")

# Effectiveness vs speed: substantive-resolution rate for the highest-volume complaint types.
topvol = df["complaint_type"].value_counts().head(10).index
eff = (df[df["complaint_type"].isin(topvol)]
       .assign(substantive=df["resolution_category"].isin(SUBSTANTIVE))
       .groupby("complaint_type", observed=True)["substantive"].mean().mul(100).round(1)
       .sort_values())
print("substantive-resolution rate, top-10 complaint types by volume (%):")
print(eff.to_string())

citywide mean = 298.2 h    median = 8.1 h    (skew -> lead with median)



complaint types with median pinned at the 6,961 h cap (report as '>cap, long-cycle'):
                  count  median
complaint_type                 
New Tree Request  20952  6960.9 



substantive-resolution rate, top-10 complaint types by volume (%):
complaint_type
Heat/Hot Water             25.6
Plumbing                   26.0
Unsanitary Condition       31.6
Noise - Residential        31.8
Abandoned Vehicle          37.0
Blocked Driveway           40.5
Illegal Parking            40.5
Noise - Street/Sidewalk    44.8
Street Condition           47.0
Noise - Commercial         51.4


## 12. Export the cleaned dataset

Written two ways:

- **`data/311_cleaned_2024.parquet`** (~83 MB) — the **committed** artifact and the source
  for the SQL layer and Power BI. Parquet's columnar + zstd compression shrinks the highly
  repetitive agency text from ~961 MB to comfortably under GitHub's 100 MB limit, with **no
  loss of rows or columns**. Both pandas and Power BI read it natively.
- **`data/311_cleaned_2024.csv`** (~961 MB) — a local, human-readable copy (gitignored).

In [15]:
OUT_PARQUET = OUT_CSV.with_suffix(".parquet")
df.to_csv(OUT_CSV, index=False)
df.to_parquet(OUT_PARQUET, compression="zstd", index=False)

print(f"wrote {OUT_CSV.name:<27} {OUT_CSV.stat().st_size / 1e6:>6,.0f} MB  (local, gitignored)")
print(f"wrote {OUT_PARQUET.name:<27} {OUT_PARQUET.stat().st_size / 1e6:>6,.0f} MB  (committed)")
print(f"{len(df):,} rows x {df.shape[1]} cols")

wrote 311_cleaned_2024.csv         1,058 MB  (local, gitignored)
wrote 311_cleaned_2024.parquet        83 MB  (committed)
3,455,782 rows x 12 cols


## Summary

Starting from 3,456,770 raw rows we:

1. Loaded the 10 analysis columns and parsed the two timestamps.
2. Derived `resolution_time_hours` — the project's core metric.
3. Kept open/unresolved tickets (null close date) for volume but excluded them from
   resolution stats.
4. Dropped the impossible negative-duration rows.
5. Winsorized the extreme right tail at the 99th percentile.
6. Standardised `borough` and `complaint_type` casing (without over-merging meaningful
   complaint subcategories).
7. Repaired the `incident_zip` float artifact.
8. Derived `resolution_category` from the agency response templates, so a fast close that
   *fixed* nothing (no evidence / gone / duplicate / wrong agency) is distinguishable from a
   real fix — and recorded the analytical caveats (median-first, cap artifact, speed ≠
   effectiveness).
9. Exported `data/311_cleaned_2024.parquet` (committed) + `.csv` (local).

**Next:** load the Parquet into SQLite and write the per-question `.sql` files (`sql/`),
leading with median resolution time and pairing it with the substantive-resolution rate.